<a href="https://colab.research.google.com/github/jamalinu/tawalt/blob/main/Tarifit_TTS_Linguistic_Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Overview
This project develops a specialized Linguistic Front-end for a Text-to-Speech (TTS) system in Tarifit (Riffian Amazigh). In speech synthesis, the front-end is responsible for converting raw, often inconsistent text into a clean, phonetically predictable format that an acoustic model can synthesize into natural-sounding speech.

Key Challenges Addressed
Orthographic Inconsistency: Tarifit is often written using various conventions (Latin, Tifinagh, or "Chat-Arabic" with numerals). This pipeline standardizes these inputs into a uniform phonetic representation.

Morphological Complexity (Clitics): Amazigh languages make heavy use of proclitics (e.g., d-, n-). Proper tokenization is crucial to ensure the TTS model manages prosodic boundaries and pauses correctly.

Phonetic Coverage: The system prepares text by mapping informal digraphs (e.g., 'gh', 'kh') to unique phonemic characters, reducing ambiguity for the downstream neural vocoder.

Technical Stack
spaCy: Used as the core NLP engine for tokenization and pipeline management.

Regex & Python: For rule-based text normalization and phonetic cleaning.

In [64]:
!pip install -q spacy phonemizer
!python -m spacy download xx_sent_ud_sm -q

import re
import unicodedata
import spacy
import pandas as pd
from collections import Counter

print("✅ Librerías instaladas para el proyecto de Voz.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 65.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Librerías instaladas para el proyecto de Voz.


In [65]:
def tarifit_text_normalizer(text):

    # 1. Minúsculas y limpieza
    text = text.lower().strip()

    # 2. De-lengthening: reduce 3+ repeticiones, preserva geminadas (tt, rr...)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 3. Pronombres → fonemas directamente (sin tags temporales)
    #    Ordenados de más largo a más corto para evitar matches parciales
    #    Ej: "nnech" debe detectarse ANTES que "nech"
    pronoun_map = {
        # Independent pronouns
        "nechchin": "neʃːin",   # 1PL  we
        "nettath":  "netːaθ",   # 3FSG she
        "kennint":  "kenːint",  # 2FPL you (fem.pl)
        "kenniw":   "kenːiw",   # 2MPL you (masc.pl)
        "nithnti":  "niθnti",   # 3FPL they (fem)
        "nithni":   "niθni",    # 3MPL they (masc)
        "netta":    "netːa",    # 3MSG he
        "nech":     "neʃ",      # 1SG  I
        "chek":     "ʃek",      # 2MSG you (masc.sg)
        "chem":     "ʃem",      # 2FSG you (fem.sg)
        # Possessive clitics (n- 'of' + pronoun → fused form)
        # Both hyphenated and non-hyphenated forms handled
        "inu":    "inu",      # POSS.1SG  mine
        "nnech":  "nːeʃ",    # POSS.2MSG yours (masc)
        "n-nem":  "nːem",    # POSS.2FSG yours (fem)
        "nnem":   "nːem",    # POSS.2FSG yours (fem) - alternate spelling
        "nnes":   "nːes",    # POSS.3SG  his/hers  (s = 3rd person marker)
        "nnegh":  "nːeɣ",    # POSS.1PL  ours
        "n-sen":  "nːsen",   # POSS.3MPL theirs (masc)
        "nsen":   "nːsen",   # POSS.3MPL theirs (masc) - alternate spelling
        "n-sent": "nːsent",  # POSS.3FPL theirs (fem)
        "nsent":  "nːsent",  # POSS.3FPL theirs (fem) - alternate spelling
    }
    for pron, fonema in sorted(pronoun_map.items(), key=lambda x: -len(x[0])):
        text = text.replace(pron, fonema)

    # 4. Diacríticos enfáticos
    diacritic_map = {
        "ṣ": "sˁ",
        "ẓ": "zˁ",
        "ḍ": "dˁ",
        "ṭ": "tˁ",
        "ḷ": "lˁ",
    }
    for old, new in diacritic_map.items():
        text = text.replace(old, new)

    # 5. Numerales chat-árabe → fonemas
    numeral_map = {
        "7": "ħ",
        "9": "q",
        "3": "ʕ",
        "8": "ɣ",
        "2": "ʔ",
    }
    for old, new in numeral_map.items():
        text = text.replace(old, new)

    # 6. Dígrafos → fonemas
    #    ORDEN IMPORTANTE: procesar dígrafos antes que letras individuales
    digraph_map = {
        "gh": "ɣ",
        "kh": "x",
        "ch": "ʃ",
        "sh": "ʃ",
        "zh": "ʒ",
        "dh": "ð",
        "th": "t",
    }
    for old, new in digraph_map.items():
        text = text.replace(old, new)

    # 7. Consonantes individuales
    text = text.replace("j", "ʒ")

    # 8. Geminadas → notación con ː
    geminate_map = {
        "tt": "tː",
        "rr": "rː",
        "ll": "lː",
        "ss": "sː",
        "nn": "nː",
        "mm": "mː",
        "ff": "fː",
        "bb": "bː",
        "dd": "dː",
        "gg": "gː",
        "zz": "zː",
    }
    for old, new in geminate_map.items():
        text = text.replace(old, new)

    # 9. Normalización de vocales informales
    vowel_map = {
        "ou": "u",
        "aa": "a",
        "ii": "i",
        "ee": "e",
    }
    for old, new in vowel_map.items():
        text = text.replace(old, new)

    # 10. Limpieza final — conserva símbolos fonéticos
    text = re.sub(r"[^\w\s\-ħqɣxʃʒɛʕðθˁːʔ]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

print("✅ Normalizador v3 listo")

✅ Normalizador v3 listo


In [66]:
# ============================================================
# CELL 4 — Script Detector + Tifinagh → Latin converter
# ============================================================

TIFINAGH_RANGE = (0x2D30, 0x2D7F)
CHAT_ARABIC_NUMERALS = set('23789')

def detect_script(text: str) -> str:
    """
    Detects the dominant writing system.
    Returns: 'tifinagh', 'chat_arabic', 'latin', or 'mixed'
    """
    if not text or not text.strip():
        return 'latin'

    chars = [c for c in text if not c.isspace()]
    total = len(chars)
    if total == 0:
        return 'latin'

    tifinagh_count = sum(1 for c in chars if TIFINAGH_RANGE[0] <= ord(c) <= TIFINAGH_RANGE[1])
    numeral_count  = sum(1 for c in chars if c in CHAT_ARABIC_NUMERALS)

    pct_tifinagh = tifinagh_count / total
    pct_numeral  = numeral_count  / total

    if pct_tifinagh > 0.5:  return 'tifinagh'
    if pct_tifinagh > 0.1:  return 'mixed'
    if pct_numeral  > 0.05: return 'chat_arabic'
    return 'latin'


TIFINAGH_TO_LATIN = {
    # Vowels
    'ⴰ': 'a',  'ⴻ': 'e',  'ⵉ': 'i',  'ⵓ': 'u',
    # Plain consonants
    'ⴱ': 'b',  'ⴷ': 'd',  'ⴼ': 'f',  'ⴳ': 'g',
    'ⵊ': 'j',  'ⴽ': 'k',  'ⵍ': 'l',  'ⵎ': 'm',
    'ⵏ': 'n',  'ⵇ': 'q',  'ⵔ': 'r',  'ⵙ': 's',
    'ⵛ': 'ch', 'ⵜ': 't',  'ⵡ': 'w',  'ⵅ': 'kh',
    'ⵢ': 'y',  'ⵣ': 'z',             # ⵢ = y (3rd person prefix)
    # Emphatics
    'ⵚ': 'ṣ',  'ⵟ': 'ṭ',  'ⵥ': 'ẓ',  'ⴹ': 'ḍ',
    # Pharyngeals and uvulars
    'ⵃ': 'ħ',  'ⵄ': 'ɛ',  'ⵖ': 'gh',
}

def tifinagh_to_latin(text: str) -> str:
    """Converts Tifinagh text to Latin transliteration."""
    return ''.join(TIFINAGH_TO_LATIN.get(c, c) for c in text)

print("✅ Script detector and Tifinagh→Latin converter ready")

✅ Script detector and Tifinagh→Latin converter ready


In [67]:
# ============================================================
# CELL 5 — Unified pipeline
# ============================================================

def process_tarifit(text: str) -> dict:
    """
    Full pipeline: detects script, converts if needed,
    and normalizes to phonemes.

    Returns a dict with:
        - script:          detected writing system
        - latin_text:      text in Latin alphabet
        - normalized_text: phonemic output for TTS
    """
    script = detect_script(text)

    if script == 'tifinagh':
        latin_text = tifinagh_to_latin(text)
    else:
        latin_text = text

    normalized = tarifit_text_normalizer(latin_text)

    return {
        'script':          script,
        'latin_text':      latin_text,
        'normalized_text': normalized
    }

print("✅ Unified pipeline ready")

✅ Unified pipeline ready


In [68]:
# Final Reflection for TTS Engineering
print("This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.")
print("By standardizing informal orthography before the acoustic model,")
print("we ensure a more natural prosody for conversational AI applications.")

This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.
By standardizing informal orthography before the acoustic model,
we ensure a more natural prosody for conversational AI applications.


In [69]:
# ============================================================
# CELL 7
# STEP — Customer Service Corpus — Tarifit
# Domain: customer support, post-sales, triage scenarios
# Designed to match the TTS deployment use case
# ============================================================

CUSTOMER_SERVICE_CORPUS = [

    # ── GREETINGS ─────────────────────────────────────────
    ("CS001", "Azul, marhba zaych, mamec ad-ggegh achk 3awnegh?",
     "Hello, welcome, how can I help you?", "greeting"),

    ("CS002", "Azul, nec da achk 3awnegh, mamec tellid?",
     "Hello, I am the support agent, how are you?", "greeting"),

    # ── COMPLAINTS ────────────────────────────────────────
    ("CS003", "war d-yiwidh ca min tugha ttargh, arzzugh ad snegh maymi.",
     "My order did not arrive, I want to know why.", "complaint"),

    ("CS004", "sarvis war ikheddem cha, arzzugh adbdigh zi jdid.",
     "The service is not working, I want to reset it.", "complaint"),

    ("CS005", "Ur zmiregh ad adfegh gha le7sab-inu",
     "I cannot access my account.", "complaint"),

    # ── CONFIRMATIONS ─────────────────────────────────────
    ("CS006", "wah, fehmegh, a-chk 3awnegh.",
     "Yes, I understand, I will help you.", "confirmation"),

    ("CS007", "raja, twarigh mani ylla lmuckil, a thhellegh lakhkhu.",
     "One moment, I see the issue, I will resolve it.", "confirmation"),

    # ── TRIAGE ────────────────────────────────────────────
    ("CS008", "Mamec da-k qqaren? mant raqm nle7sab-nech?",
     "What is your name? What is your account number?", "triage"),

    ("CS009", "Mamec ybda lmuchkil-a?",
     "How did the problem start?", "triage"),

    # ── ESCALATION ────────────────────────────────────────
    ("CS010", "Ad-ach dsse3dugh wi chek gha y3awnen, wir ttegwedh.",
     "I will transfer you to a senior agent, do not worry.", "escalation"),

    ("CS011", "Lmuchkil nnech yeqse7, anseqsa akd lfariq tiqni.",
     "Your issue is complex, we will check with the technical team.", "escalation"),
]

# ── Normalize and display ─────────────────────────────────
print("=" * 60)
print("CUSTOMER SERVICE CORPUS — TARIFIT TTS")
print("=" * 60)
print(f"Total sentences: {len(CUSTOMER_SERVICE_CORPUS)}")
print()

rows = []
for id_, sentence, gloss_en, domain in CUSTOMER_SERVICE_CORPUS:
    normalized = tarifit_text_normalizer(sentence)
    rows.append({
        "id":              id_,
        "domain":          domain,
        "original":        sentence,
        "normalized":      normalized,
        "gloss_en":        gloss_en,
    })

df_cs = pd.DataFrame(rows)
print(df_cs[["id", "domain", "original", "normalized"]].to_string(index=False))

# Export
df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print()
print("✅ Saved: tarifit_customer_service_tts.csv")

CUSTOMER SERVICE CORPUS — TARIFIT TTS
Total sentences: 11

   id       domain                                                  original                                         normalized
CS001     greeting          Azul, marhba zaych, mamec ad-ggegh achk 3awnegh?          azul marhba zayʃ mamec ad-gːeɣ aʃk ʕawneɣ
CS002     greeting                  Azul, nec da achk 3awnegh, mamec tellid?                azul nec da aʃk ʕawneɣ mamec telːid
CS003    complaint war d-yiwidh ca min tugha ttargh, arzzugh ad snegh maymi. war d-yiwið ca min tuɣa tːarɣ arzːuɣ ad sneɣ maymi
CS004    complaint         sarvis war ikheddem cha, arzzugh adbdigh zi jdid.        sarvis war ixedːem ʃa arzːuɣ adbdiɣ zi ʒdid
CS005    complaint                       Ur zmiregh ad adfegh gha le7sab-inu                   ur zmireɣ ad adfeɣ ɣa leħsab-inu
CS006 confirmation                              wah, fehmegh, a-chk 3awnegh.                             wah fehmeɣ a-ʃk ʕawneɣ
CS007 confirmation     raja, twarigh mani yll

In [70]:
# ============================================================
# STEP — Phoneme Coverage Analysis on Normalized Corpus
# Checks which phonemes from the inventory are covered
# by the normalized customer service corpus
# ============================================================


# Combine all normalized texts
all_normalized = " ".join(df_cs["normalized"].tolist())

# Clean — remove spaces and hyphens
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)

# Debug — check multi-char phonemes
print("sˁ found:", "sˁ" in ipa_clean)
print("zˁ found:", "zˁ" in ipa_clean)

# Count phoneme frequency
symbol_freq = Counter(ipa_clean)

# Check coverage against Tashelhit inventory
ALL_TASHELHIT = []
for cat in TASHELHIT_PHONEMES.values():
    for phoneme in cat.keys():
        ALL_TASHELHIT.append(phoneme)

# Tarifit orthographic equivalences
ORTHO_MAP = {
    'j': 'y',   # /j/ palatal represented as 'y' in Tarifit Latin
}

covered = []
missing = []
for p in ALL_TASHELHIT:
    if p in ipa_clean:
        covered.append(p)
    elif ORTHO_MAP.get(p) and ORTHO_MAP[p] in ipa_clean:
        covered.append(p)
    else:
        missing.append(p)
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("PHONEME COVERAGE REPORT — Customer Service Corpus")
print("=" * 50)
print(f"Total sentences       : {len(df_cs)}")
print(f"Total phonemes        : {len(ALL_TASHELHIT)}")
print(f"Phonemes covered      : {len(covered)}")
print(f"Phonemes missing      : {len(missing)}")
print(f"Coverage              : {coverage_pct:.1f}%")
print()

print("Most frequent phonemes:")
for symbol, count in symbol_freq.most_common(10):
    bar = "█" * (count // 2)
    print(f"  {symbol:<4} {count:>3}x  {bar}")

print()
if missing:
    print("Missing phonemes — add sentences to cover:")
    for p in missing:
        for cat, phonemes in TASHELHIT_PHONEMES.items():
            if p in phonemes:
                print(f"  /{p}/  [{cat}]  {phonemes[p]}")
else:
    print("✅ All phonemes covered!")

sˁ found: False
zˁ found: False
PHONEME COVERAGE REPORT — Customer Service Corpus
Total sentences       : 11
Total phonemes        : 29
Phonemes covered      : 25
Phonemes missing      : 4
Coverage              : 86.2%

Most frequent phonemes:
  a     56x  ████████████████████████████
  e     32x  ████████████████
  i     22x  ███████████
  m     21x  ██████████
  d     18x  █████████
  ɣ     18x  █████████
  n     17x  ████████
  l     15x  ███████
  r     15x  ███████
  u     12x  ██████

Missing phonemes — add sentences to cover:
  /dˁ/  [stops]  voiced emphatic dental
  /tˁ/  [stops]  voiceless emphatic dental
  /sˁ/  [fricatives]  voiceless emphatic alveolar
  /zˁ/  [fricatives]  voiced emphatic alveolar


In [71]:
# ============================================================
# CELL 9
# STEP — Add sentences to cover missing phonemes
# Target: dˁ, tˁ, sˁ, zˁ, ʃ, ʒ, ħ, ʕ, j
# ============================================================

COVERAGE_SENTENCES = [

   ("CS012", "War din bu lmuchkil. Akich achk 3awnegh lakhu.",
     "No problem. I am here to help you now.", "confirmation"),

    ("CS013", "Inay-d isem nnech l7rf s l7arf, 3afak.",
     "Tell me your name letter by letter, please.", "triage"),

    ("CS014", "Fehmegh lmuchkil i ghark.",
     "I understand your problem.", "confirmation"),

    ("CS015", "ttar tabrat di app.",
     "Request the message in the app.", "triage"),

    ("CS016", "Azwar n lmuchkil d l7sab nnech.",
     "The root of the problem is your account.", "complaint"),

    ("CS017", "Axxam ubeddi aqqa-th mujud.",
     "The support department is available.", "confirmation"),

    ("CS018", "3emmar le7sab nnech s lma3lumat.",
     "Fill your account with the data.", "triage"),

    ("CS019", "Adach dlaghigh ijj nhar nneghni.",
     "I will contact you in one day.", "escalation"),
]

# Add to main corpus and recalculate
new_rows = []
for row in COVERAGE_SENTENCES:
    id_, sentence, gloss_en, domain = row
    normalized = tarifit_text_normalizer(sentence)
    new_rows.append({
        "id": id_, "domain": domain,
        "original": sentence, "normalized": normalized,
        "gloss_en": gloss_en
    })
df_cs = pd.concat([df_cs, pd.DataFrame(new_rows)], ignore_index=True)

# Recalculate coverage
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)

covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("UPDATED COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

# Export updated CSV
df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("\n✅ CSV updated.")

UPDATED COVERAGE REPORT
Total sentences  : 19
Covered          : 25
Missing          : 4
Coverage         : 86.2%

Still missing:
  /dˁ/
  /tˁ/
  /sˁ/
  /zˁ/

✅ CSV updated.


In [72]:
# ============================================================
# CELL 10
# STEP — Final sentences to reach 100% phoneme coverage
# Native speaker validated — Jamal Saghraoui
# ============================================================

FINAL_SENTENCES = [
    # /ʒ/ — voiced postalveolar
    ("CS020", "ljihaz war ikheddem",
     "The device is not working", "complaint"),

    # /tˁ/ — voiceless emphatic dental
    ("CS021", "adhbir yedhwa deg wjenna",
     "The pigeon flew in the sky", "other"),

    # /sˁ/ — voiceless emphatic alveolar
    ("CS022", "nhara aṣmmidh",
     "Today it is cold", "other"),

    # /zˁ/ — voiced emphatic alveolar
    ("CS023", "idammen d-iẓgwaghen",
     "The blood is red", "other"),
]

new_rows = []
for row in FINAL_SENTENCES:
    id_, sentence, gloss_en, domain = row
    normalized = tarifit_text_normalizer(sentence)
    new_rows.append({
        "id": id_, "domain": domain,
        "original": sentence, "normalized": normalized,
        "gloss_en": gloss_en
    })
df_cs = pd.concat([df_cs, pd.DataFrame(new_rows)], ignore_index=True)

# Recalculate
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)
covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("FINAL COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("✅ CSV updated.")

FINAL COVERAGE REPORT
Total sentences  : 23
Covered          : 27
Missing          : 2
Coverage         : 93.1%

Still missing:
  /dˁ/
  /tˁ/
✅ CSV updated.


In [73]:
# CELL 11
# Final sentence — covers /j/ and /tˁ/
df_cs = pd.concat([df_cs, pd.DataFrame([{
    "id": "CS024",
    "domain": "other",
    "original": "y-uyar ghar thaddath",
    "normalized": tarifit_text_normalizer("y-uyar ghar thaddath"),
    "gloss_en": "he went home"
}])], ignore_index=True)

# Recalculate
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)
covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("FINAL COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("✅ CSV updated.")

FINAL COVERAGE REPORT
Total sentences  : 24
Covered          : 27
Missing          : 2
Coverage         : 93.1%

Still missing:
  /dˁ/
  /tˁ/
✅ CSV updated.


In [74]:
from google.colab import files
files.download("tarifit_customer_service_tts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Linguistic Notes (documentation)

In [75]:
# ============================================================
# STEP — Tashelhit (Souss Amazigh) Phoneme Inventory
# ISO 639-3: shi | Region: Anti-Atlas, Souss Valley, Morocco
# Comparison with Tarifit to identify TTS adaptation needs
# ============================================================

TASHELHIT_PHONEMES = {

    'vowels': {
        'a': 'open central unrounded',
        'i': 'close front unrounded',
        'u': 'close back rounded',
        # Tashelhit allows syllabic consonants — no schwa needed
    },

    'stops': {
        'b':  'voiced bilabial',
        'd':  'voiced dental',
        'dˁ': 'voiced emphatic dental',
        'g':  'voiced velar',
        'k':  'voiceless velar',
        'q':  'voiceless uvular',
        't':  'voiceless dental',
        'tˁ': 'voiceless emphatic dental',
    },

    'fricatives': {
        'f':  'voiceless labiodental',
        's':  'voiceless alveolar',
        'sˁ': 'voiceless emphatic alveolar',
        'z':  'voiced alveolar',
        'zˁ': 'voiced emphatic alveolar',
        'ʃ':  'voiceless postalveolar',
        'ʒ':  'voiced postalveolar',      # present in Tashelhit, rare in Tarifit
        'x':  'voiceless velar',
        'ɣ':  'voiced velar fricative',
        'h':  'voiceless glottal',
        'ħ':  'voiceless pharyngeal',
        'ʕ':  'voiced pharyngeal',
    },

    'nasals': {
        'm': 'bilabial nasal',
        'n': 'alveolar nasal',
    },

    'liquids': {
        'l': 'alveolar lateral',
        'r': 'alveolar trill',
    },

    'glides': {
        'w': 'labial-velar glide',
        'y': 'palatal glide',    # palatal glide — Tarifit orthography
    },
}

# ── Key differences vs Tarifit ────────────────────────────
TARIFIT_ONLY  = ['tʃ']        # affricate present in Tarifit, absent in Tashelhit
TASHELHIT_ONLY = ['ʒ']        # voiced postalveolar, more frequent in Tashelhit

print("=" * 50)
print("TASHELHIT PHONEME INVENTORY")
print("=" * 50)
total = sum(len(v) for v in TASHELHIT_PHONEMES.values())
print(f"Total phonemes: {total}")
print()
for cat, phonemes in TASHELHIT_PHONEMES.items():
    symbols = "  ".join(phonemes.keys())
    print(f"{cat.upper():<14}: {symbols}")

print()
print("KEY DIFFERENCES vs TARIFIT:")
print(f"  Tarifit only  : {TARIFIT_ONLY}  (affricate tʃ)")
print(f"  Tashelhit only: {TASHELHIT_ONLY}  (voiced postalveolar ʒ)")
print()
print("CRITICAL FOR TTS:")
print("  → Tashelhit allows syllabic consonants (e.g. /tfkt/ = 'you closed')")
print("  → Most neural vocoders need explicit vowel insertion rules")

TASHELHIT PHONEME INVENTORY
Total phonemes: 29

VOWELS        : a  i  u
STOPS         : b  d  dˁ  g  k  q  t  tˁ
FRICATIVES    : f  s  sˁ  z  zˁ  ʃ  ʒ  x  ɣ  h  ħ  ʕ
NASALS        : m  n
LIQUIDS       : l  r
GLIDES        : w  y

KEY DIFFERENCES vs TARIFIT:
  Tarifit only  : ['tʃ']  (affricate tʃ)
  Tashelhit only: ['ʒ']  (voiced postalveolar ʒ)

CRITICAL FOR TTS:
  → Tashelhit allows syllabic consonants (e.g. /tfkt/ = 'you closed')
  → Most neural vocoders need explicit vowel insertion rules
